# Centauri Bloom — FineWeb-Edu + TinyStories
75% FineWeb-Edu / 25% TinyStories by target token budget; compare source-specific loss and generation before merging.

**Standard workflow:** select GPU, run setup/configuration, authorize with the password button, then run remaining cells. Current HF main is pinned before dataset download. Publication uses a pull request only. Review and merge the PR before another run. Set RECOVER_RUN_DIR to the prior run folder to recover saved windows after a runtime restart before the first training checkpoint. Never put tokens in code.

In [ ]:
# 1. Clone Centauri Bloom (main) and install dependencies
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = Path('/kaggle/working').is_dir()
WORKSPACE = Path('/content') if IN_COLAB else (Path('/kaggle/working') if IN_KAGGLE else Path.cwd())
if Path('pyproject.toml').is_file() and Path('bloom/model.py').is_file():
    REPO = Path.cwd()
else:
    REPO = WORKSPACE / 'centauri-bloom'
    if not (REPO / 'pyproject.toml').is_file():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/InterstellarNerd8760/centauri-bloom.git', str(REPO)], check=True)
os.chdir(REPO)
if (REPO/'.git').exists() and subprocess.check_output(['git','branch','--show-current'],text=True).strip()=='main' and not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],text=True).strip():
    subprocess.run(['git','pull','--ff-only','origin','main'],check=True)
os.environ['HF_HUB_DISABLE_XET']='1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers', 'datasets', 'huggingface_hub', 'ipywidgets'], check=True)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
subprocess.run([sys.executable,'-c','from bloom.notebook_workflow import preflight; preflight()'],check=True)
from bloom.notebook_workflow import preflight, parent_total, save_corpus, restore_corpus, cached_revision, submit_candidate, sha256
preflight()


In [ ]:
# 2. Configure run & choose persistent output
import json, torch
from pathlib import Path

from datetime import datetime, timezone
RUN_NAME = 'bloom-fineweb-edu-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
HF_REPO_ID = 'Centauri-Bloom/Bloom-Text-0.1'
USE_TINYSTORIES = True # Replay previously trained language data

SUBMIT_HF_PR = True  # PR only; never direct main commits
RECOVER_RUN_DIR = ''  # Prior run folder to recover pre-training data
STEPS = 1500                  # Additional optimizer steps; adjust after evaluation
BATCH_SIZE = 16
GRAD_ACCUM = 2
LEARNING_RATE = 5e-5
WARMUP_STEPS = 50
SAVE_EVERY = 250
EVAL_BATCHES = 32
SEED = 46


IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = Path('/kaggle/working').is_dir()
SAVE_TO_DRIVE = IN_COLAB
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/centauri-bloom-runs') / RUN_NAME
elif IN_KAGGLE:
    OUT_DIR = Path('/kaggle/working/bloom-runs') / RUN_NAME
else:
    OUT_DIR = Path(os.environ.get('BLOOM_OUTPUT_DIR', str(REPO / 'runs'))) / RUN_NAME
if RECOVER_RUN_DIR:
    OUT_DIR=Path(RECOVER_RUN_DIR).expanduser()
    RUN_NAME=OUT_DIR.name
    if not (OUT_DIR/'processed-corpus.npz').is_file() or not (OUT_DIR/'data-provenance.json').is_file():
        raise RuntimeError('Recovery requires cached corpus and provenance.')
    if (OUT_DIR/'metrics.json').exists():
        raise RuntimeError('Training checkpoint already exists. Preserve it; use upload recovery or a new run.')
else:
    OUT_DIR.mkdir(parents=True,exist_ok=False)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('Select an authorized GPU runtime before training.')
print('Device:', DEVICE, '| Output dir:', OUT_DIR)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
preflight(DEVICE)
settings={'track':"bloom-fineweb-edu-colab",'steps':STEPS,'batch_size':BATCH_SIZE,'grad_accum':GRAD_ACCUM,'seed':SEED,'learning_rate':LEARNING_RATE}
settings_path=OUT_DIR/'run-settings.json'
if RECOVER_RUN_DIR and settings_path.exists() and json.loads(settings_path.read_text())!=settings:
    raise RuntimeError('Recovery settings differ from original run.')
if RECOVER_RUN_DIR and not settings_path.exists():
    print('Legacy corpus cache: verify this is the correct notebook and original settings.')
settings_path.write_text(json.dumps(settings,indent=2))


In [ ]:
# 3. Authorize candidate submission
from huggingface_hub import HfApi
from bloom.notebook_workflow import auth_client
AUTH_API = None
if not SUBMIT_HF_PR:
    AUTH_API = HfApi()
    print('Local-only run: no upload.')
elif IN_COLAB:
    from IPython.display import display
    import ipywidgets as widgets
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
    token_field=widgets.Password(description='HF token:',placeholder='Paste token')
    authorize=widgets.Button(description='Authorize upload',button_style='success')
    auth_out=widgets.Output()
    def authenticate(_):
        global AUTH_API
        with auth_out:
            auth_out.clear_output()
            try:
                AUTH_API=auth_client(HF_REPO_ID,token_field.value)
                token_field.value=''
                authorize.disabled=True
                print('Authenticated. Candidate PR only. HF checks write permissions on submission.')
            except Exception as exc:
                print('Authentication failed:',type(exc).__name__)
    authorize.on_click(authenticate)
    display(token_field,authorize,auth_out)
else:
    from getpass import getpass
    token=os.environ.get('HF_TOKEN')
    if not token and IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            token=UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            pass
    AUTH_API=auth_client(HF_REPO_ID,token or getpass('HF token: '))
    del token


In [ ]:
# 4. Pin the latest parent; prepare or recover corpus
import hashlib,subprocess,random,shutil
from itertools import islice
from tokenizers import Tokenizer
from datasets import load_dataset
from huggingface_hub import HfApi,snapshot_download
if AUTH_API is None:
    raise RuntimeError('Authorize upload first.')
PARENT_REVISION = AUTH_API.model_info(HF_REPO_ID).sha
if not PARENT_REVISION:
    raise RuntimeError('Could not resolve parent revision')
PARENT_DIR = Path(snapshot_download(repo_id=HF_REPO_ID, revision=PARENT_REVISION, token=AUTH_API.token,
    allow_patterns=['config.json','tokenizer.json','model.safetensors','metrics.json']))
parent_metrics = json.loads((PARENT_DIR / 'metrics.json').read_text())
cfg_raw = json.loads((PARENT_DIR / 'config.json').read_text())
VOCAB_SIZE = cfg_raw['vocab_size']
CONTEXT = cfg_raw['context']
tok_path = OUT_DIR / 'tokenizer.json'
shutil.copyfile(PARENT_DIR / 'tokenizer.json', tok_path)
tokenizer = Tokenizer.from_file(str(tok_path))
if tokenizer.get_vocab_size() != VOCAB_SIZE:
    raise ValueError('Parent tokenizer/config mismatch')
eot_id = tokenizer.token_to_id('<|endoftext|>')
if eot_id is None:
    raise ValueError('Missing original end-of-text token')
print('Parent revision:', PARENT_REVISION, '| original vocab:', VOCAB_SIZE)
print('Reported parent training step:', parent_metrics.get('step'))
if RECOVER_RUN_DIR:
    if cached_revision(OUT_DIR/'processed-corpus.npz')!=PARENT_REVISION:
        raise RuntimeError('Saved corpus belongs to a different parent. Start a new run instead.')
    print('Using saved token windows:',OUT_DIR/'processed-corpus.npz')
else:
    # Stream only bounded samples, never download the entire dataset.
    # Review the cards and underlying web content rights before training.
    SOURCES = [
        ('HuggingFaceFW/fineweb-edu','sample-10BT','ODC-By 1.0; underlying web content rights vary',7_500_000),
        ('roneneldan/TinyStories',None,'CDLA-Sharing-1.0 (dataset card)',2_500_000),
    ]
    train_texts, val_texts, source_manifest = [], [], []
    val_by_source = {}
    seen = set()
    api = HfApi()
    for repo_id, subset, license_note, budget in SOURCES:
        rev = api.dataset_info(repo_id).sha
        if not rev:
            raise RuntimeError('Could not pin dataset revision: '+repo_id)
        print('Streaming:', repo_id, subset, rev, flush=True)
        ds = load_dataset(repo_id, subset, split='train', revision=rev, streaming=True)
        ds = ds.shuffle(seed=SEED, buffer_size=1024)
        used = {'train':0,'validation':0}
        counts = {'train':0,'validation':0}
        records = []
        local_val = []
        val_cap = max(CONTEXT*64, budget//20)
        for row in islice(ds, 150_000):
            text = row.get('text')
            if not isinstance(text,str) or not text.strip():
                continue
            text = text.strip()
            raw_digest = hashlib.sha256(text.encode('utf-8')).hexdigest()
            ids = tokenizer.encode(text).ids[:4096]
            if len(ids)<16:
                continue
            digest = hashlib.sha256(json.dumps(ids,separators=(',',':')).encode()).hexdigest()
            if digest in seen:
                continue
            partition = 'validation' if int(digest[:8],16)%20==0 else 'train'
            cap = budget if partition=='train' else val_cap
            if used[partition] >= cap:
                if used['train']>=budget and used['validation']>=val_cap:
                    break
                continue
            seen.add(digest)
            if partition=='train':
                train_texts.append(ids)
            else:
                local_val.append(ids)
                val_texts.append(ids)
            used[partition]+=len(ids)+1
            counts[partition]+=1
            records.append({'token_ids_sha256':digest,'original_sha256':raw_digest,'split':partition,'source_id':str(row.get('id','')),'url':str(row.get('url',''))})
            if sum(counts.values())%2000==0:
                print(repo_id,used,flush=True)
        if used['train'] < budget or used['validation'] < val_cap:
            raise RuntimeError(f'Incomplete corpus for {repo_id}: {used}. Increase row limit or check stream.')
        val_by_source[repo_id]=local_val
        source_manifest.append({'dataset':repo_id,'subset':subset,'revision':rev,'license_note':license_note,'dataset_card':'https://huggingface.co/datasets/'+repo_id,'tokens':used,'documents':counts,'records':records})
        print(repo_id,used,counts,flush=True)
    random.Random(SEED).shuffle(train_texts)
    random.Random(SEED).shuffle(val_texts)
    for items in val_by_source.values():
        random.Random(SEED).shuffle(items)
    (OUT_DIR/'data-provenance.json').write_text(json.dumps(source_manifest,indent=2))
    print('Documents:',len(train_texts),len(val_texts))


In [ ]:
# 5. Pack windows and load validated parent weights
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from bloom.model import Bloom, Config
from bloom.checkpoint import config_hash, save_weights, load_weights

def encode_windows(texts, tok, context, eot):
    buf = []
    for t in texts:
        buf.extend(t)
        buf.append(eot)
    arr = np.array(buf, dtype=np.int64)
    n_win = (len(arr) - 1) // context
    if n_win < 1:
        raise ValueError(f'Only {len(arr)} tokens available; need at least {context+1}. Increase the reviewed data budget.')
    arr = arr[:n_win * context + 1]
    windows = np.lib.stride_tricks.sliding_window_view(arr, context + 1)[::context]
    x = torch.from_numpy(windows[:, :-1].copy())
    y = torch.from_numpy(windows[:, 1:].copy())
    return TensorDataset(x, y)


if RECOVER_RUN_DIR:
    train_ds,val_ds,cached_sources=restore_corpus(OUT_DIR/'processed-corpus.npz',PARENT_REVISION,CONTEXT,VOCAB_SIZE)
    train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,drop_last=True)
    val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False)
    source_val_loaders={k:DataLoader(v,batch_size=BATCH_SIZE,shuffle=False) for k,v in cached_sources.items()}
else:
    train_ds = encode_windows(train_texts, tokenizer, CONTEXT, eot_id)
    val_ds = encode_windows(val_texts, tokenizer, CONTEXT, eot_id)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    source_val_loaders = {k: DataLoader(encode_windows(v,tokenizer,CONTEXT,eot_id),batch_size=BATCH_SIZE,shuffle=False) for k,v in val_by_source.items()}
    if len(train_loader) == 0:
        raise ValueError(f'Only {len(train_ds)} train windows for batch size {BATCH_SIZE}. Increase the data budget or reduce BATCH_SIZE.')
    print(f'Train windows: {len(train_ds)} (~{len(train_ds)*CONTEXT:,} tokens) | Val windows: {len(val_ds)}')
    save_corpus(OUT_DIR/'processed-corpus.npz',train_ds,val_ds,PARENT_REVISION,source_val_loaders)
if not len(train_loader):
    raise RuntimeError('Not enough training windows for batch size.')
print('Train/validation windows:',len(train_ds),len(val_ds))
config = Config(
    vocab_size=tokenizer.get_vocab_size(),
    context=CONTEXT,
    width=cfg_raw['width'],
    heads=cfg_raw['heads'],
    layers=cfg_raw['layers'],
)
torch.manual_seed(SEED)
model = Bloom(config)
if parent_metrics.get('config_sha256') != config_hash(config):
    raise ValueError(f'Parent configuration hash mismatch at {PARENT_REVISION}: expected {parent_metrics.get("config_sha256")}, computed {config_hash(config)}. Ask maintainers to correct the published metadata.')
parent_sha = parent_metrics['weights_sha256']
model.load_state_dict(load_weights(PARENT_DIR / 'model.safetensors', parent_sha, model.state_dict()), strict=True)
model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Restored Bloom: {n_params:,} parameters | config_sha256: {config_hash(config)}')
if set(source_val_loaders)!={'HuggingFaceFW/fineweb-edu','roneneldan/TinyStories'}:
    raise RuntimeError('Missing per-source validation data in corpus cache.')


In [ ]:
# 6. Save comparable parent completions
def generate(prompt, max_new_tokens=128, temperature=0.8, top_k=40):
    model.eval()
    ids = tokenizer.encode(prompt).ids or [eot_id]
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            ctx = idx[:, -config.context:]
            logits = model(ctx)[:, -1, :] / max(temperature, 1e-5)
            if top_k:
                vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < vals[:, [-1]]] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            if nxt.item() == eot_id:
                break
            idx = torch.cat([idx, nxt], dim=1)
    return tokenizer.decode(idx[0].tolist())

PROMPTS = ['Hello', 'The Earth orbits', 'Photosynthesis is', 'Once upon a time', 'A computer program is', 'User: What is 2 plus 2?\nAssistant:']
def samples():
    results = {}
    for i,p in enumerate(PROMPTS):
        torch.manual_seed(SEED+i)
        results[p] = generate(p,max_new_tokens=48)
    return results
parent_samples = samples()
print(json.dumps(parent_samples,indent=2))


In [ ]:
# 7. Train with periodic SafeTensors checkpoints saved to Google Drive
import math, time
from itertools import islice
from torch.nn import functional as F

def evaluate(m, loader, device, max_batches):
    m.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for x, y in islice(loader, max_batches):
            x, y = x.to(device), y.to(device)
            with torch.amp.autocast('cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                logits = m(x)
                loss = F.cross_entropy(logits.flatten(0, 1), y.flatten(), reduction='sum')
            total += loss.item()
            count += y.numel()
    if not count or not math.isfinite(total):
        raise RuntimeError('Validation produced no tokens or non-finite loss; inspect the data and checkpoint before continuing.')
    return total / count, count

def save_ckpt(step, val_loss, train_loss, elapsed, tokens_seen):
    cpu_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    ckpt_file = OUT_DIR / f'step-{step:06d}.safetensors'
    sha = save_weights(ckpt_file, cpu_state, cpu_state)
    latest_file = OUT_DIR / 'latest.safetensors'
    save_weights(latest_file, cpu_state, cpu_state)
    (OUT_DIR / 'config.json').write_text(json.dumps(config.__dict__, indent=2, sort_keys=True) + '\n')
    meta = {
        'step': int(parent_metrics.get('step', 0)) + step,
        'additional_steps': step,
        'parent_revision': PARENT_REVISION,
        'parent_weights_sha256': parent_sha,
        'parent_reported_train_tokens': parent_metrics.get('train_tokens'),
        'parent_val_loss_same_windows': round(val_before, 4),
        'parent_source_val_losses': source_before,
        'optimizer_state_restored': False,
        'parameters': n_params,
        'config_sha256': config_hash(config),
        'weights_sha256': sha,
        'train_tokens': tokens_seen,
        'train_tokens_scope': 'this continuation only',
        'last_train_loss': round(train_loss, 4),
        'val_loss': round(val_loss, 4),
        'elapsed_seconds': round(elapsed, 2),
        'use_tinystories': USE_TINYSTORIES,
        'device': DEVICE,
        'run_name': RUN_NAME,
        'dataset': 'FineWeb-Edu + TinyStories replay',
        'data_provenance_sha256': hashlib.sha256((OUT_DIR/'data-provenance.json').read_bytes()).hexdigest(),
        'training_track': 'fineweb-edu-replay',
        'seed': SEED,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'grad_accum': GRAD_ACCUM,
        'code_revision': subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip(),
    }
    (OUT_DIR / 'metrics.json').write_text(json.dumps(meta, indent=2, sort_keys=True) + '\n')
    # Private run state; never load untrusted optimizer pickle files.
    torch.save({'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                'step': step, 'tokens_seen': tokens_seen, 'parent_revision': PARENT_REVISION},
               OUT_DIR / 'local-optimizer-state.pt')
    return sha

if (OUT_DIR/'metrics.json').exists():
    raise RuntimeError('Training already saved a checkpoint. Preserve it; do not rerun this cell.')
model.load_state_dict(load_weights(PARENT_DIR/'model.safetensors',parent_sha,model.state_dict()),strict=True)
torch.manual_seed(SEED)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

def lr_for_step(step):
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, STEPS - WARMUP_STEPS)
    return LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))

val_before, val_tokens = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
source_before = {k:evaluate(model,v,DEVICE,EVAL_BATCHES)[0] for k,v in source_val_loaders.items()}
print('Parent per-source losses:',source_before)
print(f'Initial val loss: {val_before:.4f} ({val_tokens} tokens)')

it = iter(train_loader)
tokens_seen = 0
t0 = time.perf_counter()
last_loss = val_before

for step in range(1, STEPS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_for_step(step - 1)
    step_loss = 0.0
    for _ in range(GRAD_ACCUM):
        batch = next(it, None)
        if batch is None:
            it = iter(train_loader)
            batch = next(it)
        x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten()) / GRAD_ACCUM
        scaler.scale(loss).backward()
        step_loss += loss.item()
        tokens_seen += y.numel()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    last_loss = step_loss

    if step == 1 or step % 100 == 0 or step == STEPS:
        dt = time.perf_counter() - t0
        print(f'step {step:4d}/{STEPS} | loss {last_loss:.4f} | {tokens_seen/max(dt,1e-6):,.0f} tok/s | {dt:.1f}s')
    if step % SAVE_EVERY == 0 or step == STEPS:
        vloss, _ = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
        sha = save_ckpt(step, vloss, last_loss, time.perf_counter() - t0, tokens_seen)
        print(f'  -> Saved checkpoint step {step} | val_loss {vloss:.4f} | sha256 {sha[:12]}...')

val_after, _ = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
print(f'Done in {time.perf_counter() - t0:.1f}s | Val loss: {val_before:.4f} -> {val_after:.4f}')
source_after = {k:evaluate(model,v,DEVICE,EVAL_BATCHES)[0] for k,v in source_val_loaders.items()}
candidate_samples = samples()
report = {'parent_revision':PARENT_REVISION,'parent_source_val_losses':source_before,'candidate_source_val_losses':source_after,'parent_samples':parent_samples,'candidate_samples':candidate_samples,'note':'Same windows for parent and candidate; possible overlap with earlier training. Samples are not scored benchmarks.'}
(OUT_DIR/'evaluation.json').write_text(json.dumps(report,indent=2))
metrics = json.loads((OUT_DIR/'metrics.json').read_text())
metrics['source_val_losses'] = source_after
metrics['evaluation_sha256'] = hashlib.sha256((OUT_DIR/'evaluation.json').read_bytes()).hexdigest()
(OUT_DIR/'metrics.json').write_text(json.dumps(metrics,indent=2,sort_keys=True)+'\n')
print(json.dumps(report,indent=2))

metrics=json.loads((OUT_DIR/'metrics.json').read_text())
previous=parent_total(parent_metrics)
metrics['cumulative_train_tokens']=previous+metrics['train_tokens'] if previous is not None else None
metrics['cumulative_train_tokens_scope']='Reported prior exposures plus this run non-padding token exposures; repeated exposures included.'
metrics['step_scope']='attempted optimizer steps'
(OUT_DIR/'metrics.json').write_text(json.dumps(metrics,indent=2,sort_keys=True)+'\n')
print(json.dumps(metrics,indent=2))


In [ ]:
# 8. Submit the completed candidate. Rerun only this cell for upload retries.
if SUBMIT_HF_PR:
    if AUTH_API is None:
        raise RuntimeError('Authorize upload first.')
    submit_candidate(AUTH_API,HF_REPO_ID,OUT_DIR,STEPS,PARENT_REVISION,RUN_NAME)
else:
    print('Local candidate saved:',OUT_DIR)
